# NP-completude: as classes P, NP, co-NP e NP-completo — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. Os programas em C são compilados com `gcc` a partir do próprio
notebook: cada célula `%%writefile` grava um programa em `src/`, e a célula seguinte compila e
executa. As verificações, os gráficos e alguns exercícios ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- explicar por que a teoria usa **problemas de decisão** e resolver a **otimização** da mochila usando só um decisor;
- escrever **verificadores** polinomiais para clique, SAT e soma de subconjuntos, e comparar o custo de **verificar** com o de **resolver**;
- dar o **certificado do “não”** de um problema de co-NP (tautologia);
- implementar em C a **redução** $\textsc{Conjunto Independente} \le_P \textsc{Clique}$ e testá-la contra a força bruta;
- medir por que a PD da soma de subconjuntos é **pseudopolinomial** (NP-completude fraca);
- usar um decisor de SAT para **construir** uma atribuição (desafio).

O grafo condutor da aula é o hexágono $0-1-2-3-4-5-0$ com a corda $0-2$: ele tem o ciclo
hamiltoniano $0,1,2,3,4,5$, a clique $\{0,1,2\}$ e o conjunto independente $\{1,3,5\}$.

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess, math, random, itertools, time

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    """Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    """Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída."""
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True,
                       errors='replace')
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    """Extrai o valor inteiro de 'nome=valor' da saída."""
    for tok in saida.split():
        if tok.startswith(nome + '='):
            return int(tok.split('=', 1)[1])
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

# Grafo condutor: hexágono 0-1-2-3-4-5-0 com a corda 0-2.
HEXA = (6, [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0), (0, 2)])
# Grafo de Petersen: ciclo externo 0..4, estrela interna 5..9, raios i -- i+5.
PETERSEN = (10, [(i, (i + 1) % 5) for i in range(5)] +
                [(5 + i, 5 + (i + 2) % 5) for i in range(5)] +
                [(i, i + 5) for i in range(5)])

def entrada_grafo(n, arestas, extra=''):
    """Formato lido pelos programas em C: 'n m', m linhas 'u v' e, opcionalmente, mais dados."""
    linhas = [f'{n} {len(arestas)}'] + [f'{u} {v}' for u, v in arestas]
    return '\n'.join(linhas) + '\n' + extra

def matriz(n, arestas):
    adj = [[False] * n for _ in range(n)]
    for u, v in arestas:
        adj[u][v] = adj[v][u] = True
    return adj

def grafo_aleatorio(n, p, rng):
    return [(u, v) for u in range(n) for v in range(u + 1, n) if rng.random() < p]

In [ ]:
# Desenho de grafos pequenos: vértices destacados em laranja, arestas destacadas grossas.
import matplotlib.pyplot as plt

def desenha(n, arestas, pos=None, vdest=(), adest=(), titulo='', ax=None):
    if pos is None:
        pos = {v: (math.cos(math.pi / 2 + 2 * math.pi * v / n),
                   math.sin(math.pi / 2 + 2 * math.pi * v / n)) for v in range(n)}
    adest = {frozenset(a) for a in adest}
    if ax is None:
        _, ax = plt.subplots(figsize=(3, 3))
    for u, v in arestas:
        d = frozenset((u, v)) in adest
        ax.plot(*zip(pos[u], pos[v]), color='darkorange' if d else 'gray',
                lw=3 if d else 1, zorder=1)
    for v, (x, y) in pos.items():
        ax.scatter(x, y, s=330, zorder=3, edgecolor='steelblue',
                   color='navajowhite' if v in vdest else 'lightsteelblue')
        ax.text(x, y, str(v), ha='center', va='center', fontsize=9, zorder=4)
    ax.set_title(titulo, fontsize=10); ax.axis('off'); ax.set_aspect('equal'); ax.margins(0.12)
    return ax

POS_PETERSEN = {}
for i in range(5):
    a = math.pi / 2 + 2 * math.pi * i / 5
    POS_PETERSEN[i] = (math.cos(a), math.sin(a))
    POS_PETERSEN[i + 5] = (0.5 * math.cos(a), 0.5 * math.sin(a))

fig, axs = plt.subplots(1, 2, figsize=(6.5, 3))
desenha(*HEXA, vdest={1, 3, 5}, adest=[(0, 1), (1, 2), (0, 2)],
        titulo='grafo condutor: clique {0,1,2},\nindependente {1,3,5}', ax=axs[0])
desenha(*PETERSEN, pos=POS_PETERSEN, titulo='grafo de Petersen', ax=axs[1])
plt.show()

## 1. Problemas de decisão: otimizar usando só um decisor

A teoria trata de problemas de **decisão** (resposta sim/não). A versão de decisão da mochila
pergunta: *existe um subconjunto de peso $\le W$ e valor $\ge V$?* Se soubermos otimizar, sabemos
decidir (compare o ótimo com $V$). O caminho inverso também vale: com um decisor e uma **busca
binária** em $V$, achamos o valor ótimo com $O(\log \sum v_i)$ chamadas, um número polinomial no
tamanho da entrada.

A célula abaixo traz um decisor pronto (por dentro ele usa a PD da Unidade 2, mas trate-o como uma
caixa-preta) e mostra quantos bits ocupa a capacidade $W$: a PD custa $\Theta(nW)$, e $W$ cresce
exponencialmente com o número de bits.

In [ ]:
# Exemplo: o decisor da mochila (caixa-preta) e o tamanho de W em bits.

CHAMADAS = 0

def mochila_decisao(itens, W, V):
    """Existe subconjunto de itens (peso, valor) com peso <= W e valor >= V?"""
    global CHAMADAS
    CHAMADAS += 1
    melhor = [0] * (W + 1)                    # PD da Unidade 2 com vetor único
    for w, v in itens:
        for c in range(W, w - 1, -1):
            melhor[c] = max(melhor[c], melhor[c - w] + v)
    return melhor[W] >= V

# Instância da Unidade 2: W = 5, itens (peso, valor); ótimo {1, 2, 4} com valor 37.
ITENS = [(2, 12), (1, 10), (3, 20), (2, 15)]
print('valor >= 37?', mochila_decisao(ITENS, 5, 37))
print('valor >= 38?', mochila_decisao(ITENS, 5, 38))

for W in (5, 5_000, 5_000_000):
    print(f'W = {W:>9}: {W.bit_length():2} bits, PD com {len(ITENS) * W:>9} células')

In [ ]:
# Exercício 1: otimização a partir da decisão
# TODO: devolva o MAIOR valor V tal que mochila_decisao(itens, W, V) é verdadeiro,
#       usando busca binária em V no intervalo [0, soma dos valores].
#       Use apenas mochila_decisao (não reimplemente a PD).

def mochila_otimo(itens, W):
    # TODO: implemente aqui
    pass

# Teste automático
CHAMADAS = 0
assert mochila_otimo(ITENS, 5) == 37, 'Verifique sua implementação (instância da Unidade 2: 37)'
assert CHAMADAS <= 8, f'Use busca binária: {CHAMADAS} chamadas é demais'
assert mochila_otimo([(10, 60), (20, 100), (30, 120)], 50) == 220, 'Verifique (CLRS Fig. 16.2: 220)'
assert mochila_otimo([(3, 5)], 2) == 0, 'Verifique o caso em que nada cabe'
rng = random.Random(1)
for _ in range(100):
    itens = [(rng.randint(1, 10), rng.randint(1, 30)) for _ in range(rng.randint(1, 6))]
    W = rng.randint(0, 25)
    certo = max(sum(v for _, v in s) for k in range(len(itens) + 1)
                for s in itertools.combinations(itens, k) if sum(w for w, _ in s) <= W)
    assert mochila_otimo(itens, W) == certo, f'erro em {itens}, W={W}'
print('Exercício 1 ok!')

## 2. A classe NP: verificar é fácil, procurar é caro

Um problema está em **NP** quando toda instância *sim* tem um **certificado** curto que um
**verificador** confere em tempo polinomial. Para o ciclo hamiltoniano, o certificado é a
sequência dos vértices; o verificador confere em $\Theta(n)$ que cada vértice aparece uma vez e que
vértices consecutivos são vizinhos.

O programa abaixo tem dois modos: `verifica` lê um ciclo e chama o verificador uma vez; `busca`
gera **todas** as permutações com o vértice $0$ fixo e chama o verificador em cada uma, até achar
um ciclo. É a força bruta “gerar e verificar”, que transforma qualquer verificador num algoritmo
exponencial.

In [ ]:
%%writefile src/ham.c
/* Ciclo hamiltoniano: verificador Theta(n) e busca por forca bruta.
   Entrada: "n m", m linhas "u v" (grafo nao dirigido) e, no modo verifica, n vertices.
   Uso: ./ham verifica   ou   ./ham busca */
#include <stdio.h>
#include <string.h>

#define MAXN 16
int n, adj[MAXN][MAXN];
long long testados = 0;

/* Devolve 1 se c[0..n-1] e um ciclo hamiltoniano do grafo. */
int verificaHam( int c[]) {
   int visto[MAXN] = {0};
   for (int i = 0; i < n; ++i) {
      int u = c[i], v = c[(i + 1) % n];
      if (u < 0 || u >= n || visto[u]) return 0;
      visto[u] = 1;
      if (!adj[u][v]) return 0;
   }
   return 1;
}

/* Gera as permutacoes de c[k..n-1] (c[0] = 0 fixo) e verifica cada uma. */
int busca( int c[], int k) {
   if (k == n) { ++testados;  return verificaHam( c); }
   for (int i = k; i < n; ++i) {
      int t = c[k]; c[k] = c[i]; c[i] = t;
      if (busca( c, k + 1)) return 1;
      t = c[k]; c[k] = c[i]; c[i] = t;
   }
   return 0;
}

int main( int argc, char *argv[]) {
   int m, c[MAXN];
   if (scanf( "%d %d", &n, &m) != 2) return 1;
   for (int i = 0; i < m; ++i) {
      int u, v;
      if (scanf( "%d %d", &u, &v) != 2) return 1;
      adj[u][v] = adj[v][u] = 1;
   }
   if (argc > 1 && strcmp( argv[1], "verifica") == 0) {
      for (int i = 0; i < n; ++i) if (scanf( "%d", &c[i]) != 1) return 1;
      printf( "valido=%d\n", verificaHam( c));
      return 0;
   }
   for (int i = 0; i < n; ++i) c[i] = i;
   int achou = busca( c, 1);
   printf( "ham=%d testados=%lld\n", achou, testados);
   if (achou) { for (int i = 0; i < n; ++i) printf( "%d ", c[i]);  printf( "\n"); }
   return 0;
}

In [ ]:
# Exemplo: verificar o ciclo dos slides, buscar um ciclo e tentar no grafo de Petersen.
rodar('ham.c', ['verifica'], entrada_grafo(*HEXA, '0 1 2 3 4 5\n'))
rodar('ham.c', ['verifica'], entrada_grafo(*HEXA, '0 2 1 3 4 5\n'))   # 1-3 não é aresta
rodar('ham.c', ['busca'], entrada_grafo(*HEXA))
rodar('ham.c', ['busca'], entrada_grafo(*PETERSEN))                   # testa as 9! permutações

In [ ]:
# Visualização: tempo da força bruta x tempo de UMA verificação.
# Grafo sem ciclo hamiltoniano (completo em 0..n-2 e o vértice n-1 pendurado em 0):
# a busca testa todas as (n-1)! permutações.

def sem_ciclo(n):
    return n, [(u, v) for u in range(n - 1) for v in range(u + 1, n - 1)] + [(0, n - 1)]

ns, t_busca, t_verif = [], [], []
for n in range(6, 12):
    ini = time.perf_counter(); saida = rodar('ham.c', ['busca'], entrada_grafo(*sem_ciclo(n)), mostrar=False)
    t_busca.append(time.perf_counter() - ini)
    ini = time.perf_counter(); rodar('ham.c', ['verifica'], entrada_grafo(*sem_ciclo(n), ' '.join(map(str, range(n)))), mostrar=False)
    t_verif.append(time.perf_counter() - ini)
    ns.append(n)
    print(f'n = {n:2}: {campo(saida, "testados"):>9} permutações testadas')

plt.figure(figsize=(5, 3))
plt.semilogy(ns, t_busca, 'o-', label='busca (gerar e verificar)')
plt.semilogy(ns, t_verif, 's-', label='uma verificação')
plt.xlabel('n'); plt.ylabel('tempo (s, escala log)'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Verificar x procurar'); plt.show()
# A verificação é dominada pelo custo de iniciar o processo; a busca cresce como (n-1)!.

### Exercício 2: verificadores

Escreva os três verificadores abaixo. Cada um recebe a instância e o **certificado** e devolve
`True` só se o certificado prova que a resposta é sim. Atenção: um verificador precisa
**rejeitar** certificados malformados (vértices repetidos, índices fora do intervalo, tamanho
errado), senão um “certificado” falso enganaria o verificador.

- `verifica_clique(n, arestas, k, S)`: `S` é uma lista de $k$ vértices distintos de $0..n-1$, dois a dois vizinhos.
- `verifica_sat(clausulas, atrib)`: cada cláusula é uma lista de inteiros não nulos ($3$ é $x_3$, $-3$ é $\lnot x_3$); `atrib[i]` é o valor de $x_i$ (`atrib[0]` não é usado). Toda cláusula precisa de um literal verdadeiro.
- `verifica_soma(numeros, t, indices)`: `indices` é uma lista de posições distintas de `numeros` cuja soma é $t$.

In [ ]:
# Exercício 2: verificadores polinomiais
# TODO: implemente os três verificadores descritos acima.

def verifica_clique(n, arestas, k, S):
    # TODO: implemente aqui
    pass

def verifica_sat(clausulas, atrib):
    # TODO: implemente aqui
    pass

def verifica_soma(numeros, t, indices):
    # TODO: implemente aqui
    pass

# Teste automático
n, A = HEXA
assert verifica_clique(n, A, 3, [0, 1, 2]) is True, 'Verifique: {0,1,2} é clique no grafo condutor'
assert verifica_clique(n, A, 3, [1, 3, 5]) is False, 'Verifique: {1,3,5} não é clique'
assert verifica_clique(n, A, 3, [0, 0, 1]) is False, 'Verifique: vértices repetidos'
assert verifica_clique(n, A, 3, [0, 1]) is False, 'Verifique: tamanho errado'
assert verifica_clique(n, A, 2, [5, 6]) is False, 'Verifique: vértice fora do intervalo'
PHI = [[1, -2], [-1, 2, 3], [-3, -1]]          # a fórmula dos slides
assert verifica_sat(PHI, [None, False, False, True]) is True, 'Verifique: x1=0, x2=0, x3=1 satisfaz'
assert verifica_sat(PHI, [None, True, True, True]) is False, 'Verifique: a terceira cláusula falha'
assert verifica_soma([3, 5, 6, 7], 15, [0, 1, 3]) is True, 'Verifique: 3 + 5 + 7 = 15'
assert verifica_soma([3, 5, 6, 7], 15, [1, 1, 1]) is False, 'Verifique: índices repetidos'
assert verifica_soma([3, 5, 6, 7], 15, [0, 1, 4]) is False, 'Verifique: índice fora do intervalo'
print('Exercício 2 ok!')

## 3. A classe co-NP: o certificado do “não”

$\textsc{Tautologia}$ pergunta se uma fórmula é verdadeira para **toda** atribuição. Não se
conhece certificado curto para o “sim”, mas o “não” tem um: uma atribuição que torna a fórmula
falsa. Por isso $\textsc{Tautologia} \in$ co-NP. Do mesmo modo, a busca da seção 2 testou as
$9! = 362\,880$ permutações do grafo de Petersen para dizer “não tem ciclo hamiltoniano”, e
ninguém conhece um certificado curto para essa resposta em geral.

Abaixo, as fórmulas são funções Python de $n$ variáveis booleanas. A busca do contraexemplo é
exponencial; **conferir** um contraexemplo custa uma única avaliação.

In [ ]:
# Exemplo: tautologia por força bruta; o contraexemplo é o certificado do "não".

def contraexemplo(formula, n):
    """Devolve uma atribuição (tupla) que torna a fórmula falsa, ou None se ela é tautologia."""
    for atrib in itertools.product([False, True], repeat=n):
        if not formula(*atrib):
            return atrib
    return None

def implica(a, b):
    return (not a) or b

terceiro_excluido = lambda p: p or not p
de_morgan = lambda p, q: (not (p and q)) == ((not p) or (not q))
silogismo = lambda p, q, r: implica(implica(p, q) and implica(q, r), implica(p, r))
reciproca = lambda p, q: implica(implica(p, q), implica(q, p))

for nome, f, n in [('p ou não p', terceiro_excluido, 1), ('De Morgan', de_morgan, 2),
                   ('silogismo', silogismo, 3), ('recíproca', reciproca, 2)]:
    c = contraexemplo(f, n)
    print(f'{nome:12}:', 'tautologia' if c is None else f'NÃO é tautologia; contraexemplo {c}')

# Conferir o certificado do "não" é uma avaliação só:
print('confere:', reciproca(*contraexemplo(reciproca, 2)) is False)

**Para pensar.** O teorema do fluxo máximo e corte mínimo (aula passada) dá certificados para os
dois lados de “o fluxo máximo é $\ge k$?”: um fluxo de valor $k$ para o sim, um corte de
capacidade $< k$ para o não. Por isso esse problema está em NP $\cap$ co-NP (e, de fato, em P).
Que certificado do “não” você daria para “o grafo bipartido tem emparelhamento perfeito?”
(Dica: teorema de Hall.)

## 4. Redução polinomial: $\textsc{Conjunto Independente} \le_P \textsc{Clique}$

$S$ é independente em $G$ se e só se $S$ é clique no **complemento** $\overline{G}$, o grafo que
liga exatamente os pares que **não** são vizinhos em $G$. Então $f(G, k) = (\overline{G}, k)$ é
uma redução: a resposta é a mesma, e montar $\overline{G}$ custa $O(n^2)$.

O programa abaixo já tem um decisor de clique por força bruta (`temClique`, que testa os
subconjuntos). No modo `indep`, ele deve responder se $G$ tem conjunto independente de tamanho
$k$ **sem** um algoritmo novo: só chamando `complemento` e depois `temClique`.

In [ ]:
# Exemplo: o grafo condutor e o seu complemento.
n, A = HEXA
adj = matriz(n, A)
COMP = [(u, v) for u in range(n) for v in range(u + 1, n) if not adj[u][v]]
print('arestas do complemento:', COMP)
fig, axs = plt.subplots(1, 2, figsize=(6.5, 3))
desenha(n, A, vdest={1, 3, 5}, titulo='G: {1,3,5} independente', ax=axs[0])
desenha(n, COMP, vdest={1, 3, 5}, adest=[(1, 3), (3, 5), (1, 5)], titulo='complemento: {1,3,5} é clique', ax=axs[1])
plt.show()

In [ ]:
%%writefile src/reducao.c
/* Exercicio 3: Conjunto Independente <=_P Clique pelo grafo complemento.
   Entrada: "n m", m linhas "u v" (grafo nao dirigido) e k.
   Uso: ./reducao clique   ou   ./reducao indep
   TODO: complete complemento(). */
#include <stdio.h>
#include <string.h>

#define MAXN 20
int n;

/* Ha clique de tamanho k entre os vertices v..n-1, dado que S[0..t-1] ja e clique?
   Forca bruta: cada vertice entra ou nao (2^n subconjuntos, com poda simples). */
int estende( int a[][MAXN], int S[], int t, int v, int k) {
   if (t == k) return 1;
   if (v == n || t + (n - v) < k) return 0;
   int ok = 1;
   for (int i = 0; i < t && ok; ++i) if (!a[S[i]][v]) ok = 0;
   if (ok) { S[t] = v;  if (estende( a, S, t + 1, v + 1, k)) return 1; }
   return estende( a, S, t, v + 1, k);
}

int temClique( int a[][MAXN], int k) {
   int S[MAXN];
   return estende( a, S, 0, 0, k);
}

/* Preenche comp[][] com a matriz de adjacencias do complemento de a[][] (sem lacos). */
void complemento( int a[][MAXN], int comp[][MAXN]) {
   /* TODO */
   for (int u = 0; u < n; ++u)
      for (int v = 0; v < n; ++v) comp[u][v] = 0;
}

int main( int argc, char *argv[]) {
   int m, k, adj[MAXN][MAXN] = {{0}}, comp[MAXN][MAXN];
   if (scanf( "%d %d", &n, &m) != 2) return 1;
   for (int i = 0; i < m; ++i) {
      int u, v;
      if (scanf( "%d %d", &u, &v) != 2) return 1;
      adj[u][v] = adj[v][u] = 1;
   }
   if (scanf( "%d", &k) != 1) return 1;
   if (argc > 1 && strcmp( argv[1], "indep") == 0) {
      complemento( adj, comp);                       /* a reducao f */
      printf( "indep=%d\n", temClique( comp, k));
   } else
      printf( "clique=%d\n", temClique( adj, k));
   return 0;
}

In [ ]:
# Teste automático do Exercício 3

def independente_ref(n, arestas, k):
    """Força bruta em Python: existe conjunto independente de tamanho k?"""
    adj = matriz(n, arestas)
    return any(all(not adj[u][v] for u, v in itertools.combinations(S, 2))
               for S in itertools.combinations(range(n), k))

def reducao_c(modo, n, arestas, k):
    return campo(rodar('reducao.c', [modo], entrada_grafo(n, arestas, f'{k}\n'), mostrar=False), modo)

assert reducao_c('clique', *HEXA, 3) == 1 and reducao_c('clique', *HEXA, 4) == 0, 'temClique foi alterado?'
assert reducao_c('indep', *HEXA, 3) == 1, 'Verifique sua implementação ({1,3,5} é independente)'
assert reducao_c('indep', *HEXA, 4) == 0, 'Verifique sua implementação (não há independente de tamanho 4)'
assert reducao_c('indep', *PETERSEN, 4) == 1 and reducao_c('indep', *PETERSEN, 5) == 0, \
    'Verifique sua implementação (Petersen: maior independente tem 4 vértices)'
rng = random.Random(2026)
for _ in range(150):
    n = rng.randint(1, 9)
    A = grafo_aleatorio(n, rng.random(), rng)
    k = rng.randint(1, n)
    assert reducao_c('indep', n, A, k) == independente_ref(n, A, k), f'erro em n={n}, {A}, k={k}'
print('Exercício 3 ok!')

Repare no **sentido** da redução. Ela mostra que $\textsc{Conjunto Independente}$ não é mais
difícil que $\textsc{Clique}$: um algoritmo rápido para clique daria um algoritmo rápido para
conjunto independente. Como a mesma $f$ também funciona no sentido contrário, os dois problemas
são igualmente difíceis. Na próxima aula, reduções desse tipo encadeiam
$\textsc{SAT} \le_P \textsc{3-SAT} \le_P \textsc{Clique} \le_P \cdots$

## 5. NP-completude fraca: a PD pseudopolinomial da soma de subconjuntos

$\textsc{Soma de Subconjuntos}$ (existe subconjunto de $a_1, \dots, a_n$ com soma exatamente
$t$?) é NP-completo. Mesmo assim, a PD

$$\mathit{alc}[i][s] = \mathit{alc}[i-1][s] \;\lor\; \mathit{alc}[i-1][s - a_i]$$

resolve em $\Theta(nt)$. Não há contradição: $t$ ocupa só $\lg t$ bits, e $\Theta(nt)$ é
exponencial nesse tamanho. Algoritmos assim são **pseudopolinomiais**, e problemas NP-completos
que os admitem são **fracamente** NP-completos.

### Exercício 4

Implemente `soma_subconjunto(numeros, t)` com a tabela $\mathit{alc}$ de $(n+1) \times (t+1)$
booleanos. Devolva a lista de **índices** de um subconjunto com soma $t$ (reconstruído de trás para
frente na tabela), ou `None` se não existir. Atualize a variável global `CELULAS` com o número de
células preenchidas.

In [ ]:
# Exercício 4: soma de subconjuntos por PD (pseudopolinomial)
# TODO: implemente a PD e a reconstrução do certificado.

CELULAS = 0

def soma_subconjunto(numeros, t):
    global CELULAS
    # TODO: implemente aqui
    pass

# Teste automático (usa o seu verifica_soma do Exercício 2)
r = soma_subconjunto([3, 5, 6, 7], 15)
assert r is not None and verifica_soma([3, 5, 6, 7], 15, r), 'Verifique (slides: 15 = 3+5+7)'
assert soma_subconjunto([3, 5, 6, 7], 2) is None, 'Verifique: 2 não é alcançável'
assert soma_subconjunto([3, 5, 6, 7], 0) == [], 'Verifique: soma 0 é o conjunto vazio'
CELULAS = 0; soma_subconjunto([3, 5, 6, 7], 15)
assert CELULAS == 5 * 16, f'A tabela deve ter (n+1)(t+1) = 80 células, não {CELULAS}'
rng = random.Random(7)
for _ in range(200):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 8))]
    t = rng.randint(0, 60)
    existe = any(sum(s) == t for k in range(len(nums) + 1) for s in itertools.combinations(nums, k))
    r = soma_subconjunto(nums, t)
    assert (r is not None) == existe, f'resposta errada em {nums}, t={t}'
    assert r is None or verifica_soma(nums, t, r), f'certificado inválido em {nums}, t={t}'
print('Exercício 4 ok!')

In [ ]:
# Visualização: com n = 12 fixo, cada bit a mais nos números dobra a tabela.
# A força bruta testa 2^n = 4096 subconjuntos, qualquer que seja o tamanho dos números.
rng = random.Random(3)
bs, cels = [], []
for b in range(4, 15):
    nums = [rng.randint(2 ** (b - 1), 2 ** b - 1) for _ in range(12)]
    t = sum(nums) // 2
    CELULAS = 0
    soma_subconjunto(nums, t)
    bs.append(b); cels.append(CELULAS)

plt.figure(figsize=(5, 3))
plt.semilogy(bs, cels, 'o-', label='PD: (n+1)(t+1) células')
plt.semilogy(bs, [2 ** 12] * len(bs), '--', label='força bruta: 2^n subconjuntos')
plt.xlabel('bits por número'); plt.ylabel('trabalho (escala log)'); plt.grid(alpha=0.3)
plt.legend(); plt.title('PD pseudopolinomial: ótima com números pequenos'); plt.show()

## Desafio Final: da decisão à construção em SAT

O programa `sat.c` abaixo é um **decisor** de SAT por força bruta: ele responde só `sat=1` ou
`sat=0`, sem dizer qual atribuição funciona. A entrada segue o formato DIMACS simplificado:
`n m` e depois $m$ cláusulas, cada uma terminada por `0`.

Escreva `atribuicao_sat(n, clausulas)` que **constrói** uma atribuição satisfatória usando o
decisor no máximo $n + 1$ vezes (ou devolve `None` se a fórmula é insatisfazível).
*Dica:* fixe $x_1$ acrescentando a cláusula unitária $[1]$ e pergunte se a fórmula continua
satisfazível; se não, $x_1$ tem de ser falso ($[-1]$). Repita para $x_2, x_3, \ldots$

Esse fenômeno se chama **autorredutibilidade**: para problemas NP-completos, achar um certificado
não é mais difícil (a menos de um fator polinomial) que decidir. Por isso a teoria pode se limitar
a problemas de decisão. Teste com a fórmula dos slides e com fórmulas 3-SAT aleatórias de
$n = 14$ variáveis e $m = 59 \approx 4{,}2\,n$ cláusulas. Essa razão fica perto do limiar de
satisfazibilidade do 3-SAT aleatório, então aparecem fórmulas satisfazíveis e insatisfazíveis.

In [ ]:
%%writefile src/sat.c
/* Decisor de SAT por forca bruta: testa as 2^n atribuicoes.
   Entrada: "n m" e m clausulas, cada uma uma lista de literais (+i / -i) terminada por 0.
   Saida: "sat=1" ou "sat=0" (sem dizer qual atribuicao). */
#include <stdio.h>

#define MAXM 512
#define MAXL 4096
int n, m, lit[MAXL], ini[MAXM + 1];

int satisfaz( unsigned long x) {                       /* bit i-1 de x = valor de x_i */
   for (int c = 0; c < m; ++c) {
      int ok = 0;
      for (int j = ini[c]; j < ini[c + 1] && !ok; ++j) {
         int v = lit[j] > 0 ? lit[j] : -lit[j];
         int b = (x >> (v - 1)) & 1;
         ok = lit[j] > 0 ? b : !b;
      }
      if (!ok) return 0;
   }
   return 1;
}

int main( void) {
   if (scanf( "%d %d", &n, &m) != 2) return 1;
   int k = 0;
   for (int c = 0; c < m; ++c) {
      ini[c] = k;
      int l;
      while (scanf( "%d", &l) == 1 && l != 0) lit[k++] = l;
   }
   ini[m] = k;
   for (unsigned long x = 0; x < (1UL << n); ++x)
      if (satisfaz( x)) { printf( "sat=1\n");  return 0; }
   printf( "sat=0\n");
   return 0;
}

In [ ]:
# Ferramentas do desafio: o decisor (caixa-preta) e um gerador de 3-SAT aleatório.
DECISOES = 0

def decide_sat(n, clausulas):
    global DECISOES
    DECISOES += 1
    entrada = f'{n} {len(clausulas)}\n' + ''.join(' '.join(map(str, c)) + ' 0\n' for c in clausulas)
    return campo(rodar('sat.c', entrada=entrada, mostrar=False), 'sat') == 1

def tres_sat_aleatorio(n, m, rng):
    return [[v * rng.choice((1, -1)) for v in rng.sample(range(1, n + 1), 3)] for _ in range(m)]

print('fórmula dos slides é satisfazível?', decide_sat(3, [[1, -2], [-1, 2, 3], [-3, -1]]))
print('x1 e não x1?', decide_sat(1, [[1], [-1]]))

In [ ]:
# Desafio: construir a atribuição usando só o decisor
# TODO: resolva o desafio abaixo

def atribuicao_sat(n, clausulas):
    # Sua solução aqui: devolva [None, x1, x2, ..., xn] (booleanos) ou None
    pass

# Teste automático (usa o seu verifica_sat do Exercício 2)
# DECISOES = 0
# a = atribuicao_sat(3, [[1, -2], [-1, 2, 3], [-3, -1]])
# assert a is not None and verifica_sat([[1, -2], [-1, 2, 3], [-3, -1]], a) and DECISOES <= 4

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. Cap. 34 (§34.1 tempo polinomial e codificações; §34.2 verificação, NP e co-NP; §34.3 reduções e NP-completude; §34.5.2 clique; §34.5.5 soma de subconjuntos). (Em português: *Algoritmos: Teoria e Prática*, Elsevier, 2012.)
- KLEINBERG, J.; TARDOS, É. *Algorithm Design*. Addison-Wesley, 2006. §6.4 (soma de subconjuntos e mochila), §8.1, §8.3, §8.4, §8.8 e §8.9.
- LEVITIN, A. *Introduction to the Design and Analysis of Algorithms*. 3. ed. Pearson, 2012. §11.3.
- GAREY, M. R.; JOHNSON, D. S. *Computers and Intractability*. W. H. Freeman, 1979. Cap. 2 e §4.2 (NP-completude forte).
- TOSCANI, L. V.; VELOSO, P. A. S. *Complexidade de Algoritmos*. 3. ed. Bookman, 2012.
- COOK, S. A. The Complexity of Theorem-Proving Procedures. *Proc. 3rd ACM STOC*, p. 151–158, 1971.
- KARP, R. M. Reducibility among Combinatorial Problems. In: *Complexity of Computer Computations*. Plenum, 1972. p. 85–103.